# NB22 — MAG Quality Covariates

**Purpose:** Rule out that the primary PGLS signal is driven by systematic differences
in MAG quality (completeness or contamination) between metal-gene-dense and metal-gene-sparse
genera. High-quality MAGs (completeness ≥ 90%, contamination ≤ 5%) should show the
same pattern as the full dataset.

**Tests:**
1. For each genus, compute mean completeness and mean contamination from kescience_mgnify.genome
2. Two additional PGLS models with quality covariates:
   - B_std ~ density_z + completeness_z
   - B_std ~ density_z + contamination_z
3. Sensitivity: restrict to genera with mean completeness ≥ 90% AND contamination ≤ 5%

**Requires:** JupyterHub (Spark for completeness/contamination from kescience_mgnify).

**Label:** Exploratory. Run once. No iterative tuning.

**Outputs:**
- `data/mag_quality_sensitivity.csv`
- REPORT.md paragraph under Confounders / Quality sensitivity


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path('/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology')
DATA    = PROJECT / 'data'
TREE_BAC = DATA / 'gtdb_bac_genus_pruned.tree'

sys.path.insert(0, str(PROJECT / 'scripts'))
from pgls_utils import run_pgls

_SPARK_AVAILABLE = False
_spark = None
try:
    from berdl_utils import get_spark_session
    _spark = get_spark_session()
    _SPARK_AVAILABLE = True
    print('Spark OK')
except BaseException as _e:
    print(f'Spark unavailable: {_e}')

bac_base = pd.read_csv(DATA / '01_pgls_input_bacteria.csv')
print(f'Primary PGLS input: {len(bac_base)} genera')
print(f'Columns: {bac_base.columns.tolist()}')


[berdl_utils] JupyterHub SparkSession acquired: 4.0.1
Spark OK
Primary PGLS input: 1574 genera
Columns: ['genus_lower', 'ko_per_mb_primary', 'mean_genome_mb', 'mean_levins_B_std', 'phylum', 'kingdom', 'predictor_z', 'genome_mb_z']


## Block 2 — Fetch MAG quality metrics per genus

In [2]:
if not _SPARK_AVAILABLE:
    raise RuntimeError('Spark required — run in JupyterHub')

# First inspect genome table schema for quality columns
_spark.sql('DESCRIBE kescience_mgnify.genome').show(80, truncate=False)


+-----------------------+---------+-------+
|col_name               |data_type|comment|
+-----------------------+---------+-------+
|genome_id              |string   |NULL   |
|biome_id               |string   |NULL   |
|lineage                |string   |NULL   |
|is_species_rep         |boolean  |NULL   |
|species_id             |string   |NULL   |
|length                 |int      |NULL   |
|n_contigs              |int      |NULL   |
|n50                    |int      |NULL   |
|gc_content             |double   |NULL   |
|completeness           |double   |NULL   |
|contamination          |double   |NULL   |
|genome_type            |string   |NULL   |
|has_rrna_5s            |double   |NULL   |
|has_rrna_16s           |double   |NULL   |
|has_rrna_23s           |double   |NULL   |
|n_trnas                |int      |NULL   |
|genome_accession       |string   |NULL   |
|sample_accession       |string   |NULL   |
|study_accession        |string   |NULL   |
|country                |string 

In [3]:
# Quality columns are likely 'completeness' and 'contamination' (CheckM standard names)
# Adjust column names below if schema differs

quality_sql = """
    SELECT regexp_extract(lineage, 'g__([^;]+)', 1)  AS genus,
           COUNT(DISTINCT genome_id)                  AS n_mags,
           AVG(completeness)                          AS mean_completeness,
           AVG(contamination)                         AS mean_contamination,
           PERCENTILE(completeness, 0.5)              AS median_completeness,
           PERCENTILE(contamination, 0.5)             AS median_contamination
    FROM kescience_mgnify.genome
    WHERE completeness IS NOT NULL
    GROUP BY genus
"""
qual_df = _spark.sql(quality_sql).toPandas()
qual_df['genus_lower'] = qual_df['genus'].str.lower().str.strip()
qual_df = qual_df.dropna(subset=['mean_completeness','mean_contamination'])
print(f'Quality data: {len(qual_df)} genera')
print(qual_df[['mean_completeness','mean_contamination']].describe())
qual_df.to_csv(DATA / 'genus_mag_quality.csv', index=False)
print('Saved: data/genus_mag_quality.csv')


Quality data: 10855 genera
       mean_completeness  mean_contamination
count       10855.000000        10855.000000
mean           85.375079            1.503093
std             9.576807            1.057880
min            50.160000            0.000000
25%            79.764494            0.694454
50%            86.257000            1.320000
75%            92.581000            2.130000
max           100.000000            4.990000
Saved: data/genus_mag_quality.csv


## Block 3 — PGLS with quality covariates

In [ ]:

# Merge quality data
merged = bac_base.merge(qual_df[['genus_lower','mean_completeness','mean_contamination',
                                   'n_mags']], on='genus_lower', how='inner')
print(f'Genera with quality data: {len(merged)} / {len(bac_base)} from primary')

# Z-score predictors
mu, sd = merged['ko_per_mb_primary'].mean(), merged['ko_per_mb_primary'].std()
merged['ko_per_mb_z'] = (merged['ko_per_mb_primary'] - mu) / sd
cmu, csd = merged['mean_completeness'].mean(), merged['mean_completeness'].std()
merged['completeness_z'] = (merged['mean_completeness'] - cmu) / csd
xmu, xsd = merged['mean_contamination'].mean(), merged['mean_contamination'].std()
merged['contamination_z'] = (merged['mean_contamination'] - xmu) / xsd

df_fit = merged.dropna(subset=['ko_per_mb_z','mean_levins_B_std',
                                'completeness_z','contamination_z'])
print(f'Complete cases for covariate models: {len(df_fit)}')


def _extract_beta(res, focal='ko_per_mb_z'):
    """Extract focal predictor stats from single- or multi-predictor run_pgls result."""
    if 'beta' in res:
        return res['beta'], res['SE'], res['p_value']
    return res['betas'][focal], res['SEs'][focal], res['p_values'][focal]


# Baseline (single predictor)
res_base = run_pgls(df_fit, TREE_BAC, response='mean_levins_B_std',
                    predictors=['ko_per_mb_z'], taxon_col='genus_lower',
                    label='MAG_baseline', min_n=100)
base_beta, base_SE, base_p = _extract_beta(res_base)
print(f'Baseline:       β={base_beta:+.4f}, SE={base_SE:.4f}, p={base_p:.4g}, n={res_base["n"]}')

# With completeness covariate
res_comp = run_pgls(df_fit, TREE_BAC, response='mean_levins_B_std',
                    predictors=['ko_per_mb_z','completeness_z'], taxon_col='genus_lower',
                    label='MAG_completeness', min_n=100)
if isinstance(res_comp, dict) and ('beta' in res_comp or 'betas' in res_comp):
    comp_beta, comp_SE, comp_p = _extract_beta(res_comp, 'ko_per_mb_z')
    comp_beta_cov = res_comp.get('betas', {}).get('completeness_z', float('nan'))
    comp_p_cov    = res_comp.get('p_values', {}).get('completeness_z', float('nan'))
    print(f'+ completeness: β(density)={comp_beta:+.4f}, SE={comp_SE:.4f}, p={comp_p:.4g}')
    print(f'                β(completeness)={comp_beta_cov:+.4f}, p(completeness)={comp_p_cov:.4g}')
    comp_ok = True
else:
    print(f'+ completeness: FAILED — {res_comp}')
    comp_beta = comp_SE = comp_p = float('nan')
    comp_ok = False

# With contamination covariate
res_cont = run_pgls(df_fit, TREE_BAC, response='mean_levins_B_std',
                    predictors=['ko_per_mb_z','contamination_z'], taxon_col='genus_lower',
                    label='MAG_contamination', min_n=100)
if isinstance(res_cont, dict) and ('beta' in res_cont or 'betas' in res_cont):
    cont_beta, cont_SE, cont_p = _extract_beta(res_cont, 'ko_per_mb_z')
    cont_beta_cov = res_cont.get('betas', {}).get('contamination_z', float('nan'))
    cont_p_cov    = res_cont.get('p_values', {}).get('contamination_z', float('nan'))
    print(f'+ contamination: β(density)={cont_beta:+.4f}, SE={cont_SE:.4f}, p={cont_p:.4g}')
    print(f'                 β(contamination)={cont_beta_cov:+.4f}, p(contamination)={cont_p_cov:.4g}')
    cont_ok = True
else:
    print(f'+ contamination: FAILED — {res_cont}')
    cont_beta = cont_SE = cont_p = float('nan')
    cont_ok = False

# Attenuation summary
if comp_ok and cont_ok:
    att_comp = (base_beta - comp_beta) / base_beta * 100
    att_cont = (base_beta - cont_beta) / base_beta * 100
    print(f'\nβ attenuation by completeness: {att_comp:.1f}%')
    print(f'β attenuation by contamination: {att_cont:.1f}%')


## Block 4 — High-quality MAG sensitivity

In [ ]:

# Restrict to genera with mean completeness >= 90% AND mean contamination <= 5%
COMP_THRESH = 90.0
CONT_THRESH = 5.0

hq = df_fit[(df_fit['mean_completeness'] >= COMP_THRESH) &
            (df_fit['mean_contamination'] <= CONT_THRESH)].copy()
print(f'High-quality genera (completeness≥{COMP_THRESH}%, contamination≤{CONT_THRESH}%): {len(hq)}')

# Re-score density predictor z within HQ subset
mu_hq = hq['ko_per_mb_primary'].mean()
sd_hq = hq['ko_per_mb_primary'].std()
hq = hq.copy()
hq['ko_per_mb_z'] = (hq['ko_per_mb_primary'] - mu_hq) / sd_hq

res_hq = run_pgls(hq, TREE_BAC, response='mean_levins_B_std',
                  predictors=['ko_per_mb_z'], taxon_col='genus_lower',
                  label='MAG_hq_restricted', min_n=30)
hq_beta, hq_SE, hq_p = _extract_beta(res_hq)
print(f'HQ-restricted: β={hq_beta:+.4f}, SE={hq_SE:.4f}, p={hq_p:.4g}, n={res_hq["n"]}')

# Collect results
results = []

results.append({
    'model': 'baseline',
    'n_genera': res_base['n'],
    'beta': base_beta,
    'SE': base_SE,
    'p': base_p,
    'lambda_est': res_base['lambda_est'],
    'covariate': 'none',
    'completeness_threshold': None,
    'contamination_threshold': None
})

if comp_ok:
    results.append({
        'model': 'completeness_covariate',
        'n_genera': res_comp['n'],
        'beta': comp_beta,
        'SE': comp_SE,
        'p': comp_p,
        'lambda_est': res_comp['lambda_est'],
        'covariate': 'completeness_z',
        'completeness_threshold': None,
        'contamination_threshold': None
    })
else:
    print("WARNING: completeness covariate model failed - skipping")

if cont_ok:
    results.append({
        'model': 'contamination_covariate',
        'n_genera': res_cont['n'],
        'beta': cont_beta,
        'SE': cont_SE,
        'p': cont_p,
        'lambda_est': res_cont['lambda_est'],
        'covariate': 'contamination_z',
        'completeness_threshold': None,
        'contamination_threshold': None
    })
else:
    print("WARNING: contamination covariate model failed - skipping")

results.append({
    'model': 'hq_restricted',
    'n_genera': res_hq['n'],
    'beta': hq_beta,
    'SE': hq_SE,
    'p': hq_p,
    'lambda_est': res_hq['lambda_est'],
    'covariate': 'none',
    'completeness_threshold': COMP_THRESH,
    'contamination_threshold': CONT_THRESH
})

out_df = pd.DataFrame(results)
out_df.to_csv(DATA / 'mag_quality_sensitivity.csv', index=False)
print('\nSaved: data/mag_quality_sensitivity.csv')
print(out_df[['model','n_genera','beta','SE','p']].to_string(index=False))


## Block 5 — REPORT.md paragraph draft

In [8]:
print('=== REPORT.md paragraph for MAG quality sensitivity ===')
print(f"""
To rule out that the primary signal (P1: β = −0.021) is driven by MAG quality
differences between genera, we incorporated mean genome completeness and mean
contamination (computed per genus from kescience_mgnify.genome, n = [X] genera
with quality metadata) as additional covariates in the primary PGLS (NB22; exploratory).

Adding mean completeness as a covariate yielded β = [X] for metal gene density
([Y]% attenuation vs baseline); adding contamination yielded β = [X] ([Y]%
attenuation). In both cases, the metal gene density coefficient remained [significant/
directionally consistent], indicating that MAG quality differences do not account
for the observed association.

A sensitivity analysis restricted to genera with mean completeness ≥ 90% and
contamination ≤ 5% (n = [X] genera) recovered β = [X] (p = [Y]), [confirming/
weakening but maintaining the direction of] the primary result within the
highest-quality subset.
""")


=== REPORT.md paragraph for MAG quality sensitivity ===

To rule out that the primary signal (P1: β = −0.021) is driven by MAG quality
differences between genera, we incorporated mean genome completeness and mean
contamination (computed per genus from kescience_mgnify.genome, n = [X] genera
with quality metadata) as additional covariates in the primary PGLS (NB22; exploratory).

Adding mean completeness as a covariate yielded β = [X] for metal gene density
([Y]% attenuation vs baseline); adding contamination yielded β = [X] ([Y]%
attenuation). In both cases, the metal gene density coefficient remained [significant/
directionally consistent], indicating that MAG quality differences do not account
for the observed association.

A sensitivity analysis restricted to genera with mean completeness ≥ 90% and
contamination ≤ 5% (n = [X] genera) recovered β = [X] (p = [Y]), [confirming/
weakening but maintaining the direction of] the primary result within the
highest-quality subset.

